In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║   Odia-Hindi Pair Extractor — Google Colab                  ║
# ║   Reads raw indic-align CSV → outputs ory_Orya + hin_Deva   ║
# ╚══════════════════════════════════════════════════════════════╝
#
# HOW TO USE:
#   1. Run Cell 1 (this cell) — installs nothing, just imports
#   2. Run Cell 2 — upload your CSV when prompted
#   3. Run Cell 3 — extracts and downloads odia_hindi_pairs.csv

import re
import csv
import io
from google.colab import files

print('✅ Ready!')


✅ Ready!


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Load your raw CSV from Colab path
# ─────────────────────────────────────────────────────────────────

# ✏️ Change this to your file's path in Colab
FILE_PATH = '/content/drive/MyDrive/Wiki_Chat_extract.csv'   # <-- change this

with open(FILE_PATH, 'rb') as f:
    raw_bytes = f.read()

try:
    raw_text = raw_bytes.decode('utf-8')
except UnicodeDecodeError:
    raw_text = raw_bytes.decode('latin-1')

import os
filename = os.path.basename(FILE_PATH)
print(f'✅ Loaded: {filename}')
print(f'   Size: {len(raw_text):,} characters')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Extract ory_Orya + hin_Deva and save CSV
# ─────────────────────────────────────────────────────────────────

def extract_texts(cell):
    """
    Pull all quoted strings out of a cell like:
      "[array(['text one', 'text two'], dtype=object)]"
    Returns: ['text one', 'text two']
    """
    matches = re.findall(r"'((?:[^'\\]|\\.)*)'" , str(cell))
    skip = {'object', 'dtype=object', 'dtype', ''}
    return [m.strip() for m in matches if m.strip() not in skip]


def rejoin_multiline_rows(raw_text):
    """
    The raw CSV has cells that break across multiple lines.
    A real new data row always starts with [array.
    Stitch continuation lines back together.
    """
    lines = raw_text.splitlines()
    joined = []
    buffer = ''
    for line in lines:
        if buffer == '':
            buffer = line
        else:
            stripped = line.strip()
            if stripped.startswith('"[array') or stripped.startswith('[array'):
                joined.append(buffer)
                buffer = line
            else:
                buffer = buffer + ' ' + stripped
    if buffer:
        joined.append(buffer)
    return joined


# ── Rejoin rows ──────────────────────────────────────────────────
print('🔧 Processing...')
joined_rows = rejoin_multiline_rows(raw_text)

# ── Parse header ─────────────────────────────────────────────────
header_line = joined_rows[0]
all_columns = [c.strip().strip('"') for c in header_line.split(',')]
print(f'   Columns found : {all_columns}')
print(f'   Data rows     : {len(joined_rows) - 1:,}')

# ── Check required columns exist ─────────────────────────────────
TARGET_COLS = ['ory_Orya', 'hin_Deva']
missing = [c for c in TARGET_COLS if c not in all_columns]
if missing:
    print(f'\n❌ ERROR: These columns not found: {missing}')
    print(f'   Available columns: {all_columns}')
    raise SystemExit('Please check column names in your CSV header.')

ory_idx = all_columns.index('ory_Orya')
hin_idx = all_columns.index('hin_Deva')
n_cols  = len(all_columns)

# ── Extract rows ──────────────────────────────────────────────────
output_rows = []
skipped = 0

for raw_row in joined_rows[1:]:
    try:
        reader = csv.reader([raw_row])
        cells = next(reader)
    except Exception:
        skipped += 1
        continue

    if len(cells) != n_cols:
        skipped += 1
        continue

    ory_texts = extract_texts(cells[ory_idx])
    hin_texts = extract_texts(cells[hin_idx])

    max_len = max(len(ory_texts), len(hin_texts), 0)
    if max_len == 0:
        skipped += 1
        continue

    for i in range(max_len):
        ory_val = ory_texts[i] if i < len(ory_texts) else ''
        hin_val = hin_texts[i] if i < len(hin_texts) else ''
        # Only keep rows where BOTH columns have text
        if ory_val.strip() and hin_val.strip():
            output_rows.append({'ory_Orya': ory_val, 'hin_Deva': hin_val})

print(f'   Output pairs  : {len(output_rows):,}')
print(f'   Skipped rows  : {skipped:,}')

# ── Preview ───────────────────────────────────────────────────────
print('\n── Sample (first 3 pairs) ──────────────────────────────────')
for i, row in enumerate(output_rows[:3]):
    print(f"\nPair {i+1}:")
    print(f"  ory_Orya : {row['ory_Orya'][:100]}")
    print(f"  hin_Deva : {row['hin_Deva'][:100]}")

# ── Save & download ───────────────────────────────────────────────
out_filename = 'odia_hindi_pairs.csv'
buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=['ory_Orya', 'hin_Deva'])
writer.writeheader()
writer.writerows(output_rows)

with open(out_filename, 'w', encoding='utf-8-sig') as f:
    f.write(buf.getvalue())

print(f'\n💾 Saved: {out_filename}  ({len(output_rows):,} rows)')
files.download(out_filename)
print('⬇️  Download started!')
